In [1]:
import setup
setup.init_django()

In [2]:
import requests

from datetime import datetime
from urllib.parse import urlencode

from django.conf import settings
from pytz import timezone

ALPHA_VANTAGE_API_KEY = settings.ALPHA_VANTAGE_API_KEY
assert ALPHA_VANTAGE_API_KEY != ""

In [3]:
from market.models import Company, StockQuote

In [4]:
def get_alpha_vantage_company_info(symbol="AAPL"):
    args = {
        "function": "OVERVIEW",
        "apikey": ALPHA_VANTAGE_API_KEY,
        "symbol": symbol,
    }
    params = urlencode(args)
    url = f'https://www.alphavantage.co/query?{params}'
    r = requests.get(url)
    if r.status_code in range(200, 299):
        return r.json()
    return None

In [5]:
def get_company_obj(symbol="AAPL"):
    company_data = get_alpha_vantage_company_info(symbol=symbol)
    if company_data is None:
        raise Exception(f"Company does not exist. {symbol} invalid.")
    name = company_data.get("Name")
    company, created = Company.objects.update_or_create(
        ticker=symbol,
        defaults = {
            'name': name,
            'active': True
        }
    )
    return company

In [6]:
# company = get_company_obj(symbol="GOOGL")

In [7]:
# company.ticker, company.id

In [8]:
# month = datetime.now().strftime("%Y-%m")
# month

In [9]:
def get_alpha_vantage_company_stock_quotes(symbol="AAPL", month=None):
    if month is None:
        month = datetime.now().strftime("%Y-%m")
    args = {
        "function": "TIME_SERIES_INTRADAY",
        "apikey": ALPHA_VANTAGE_API_KEY,
        "symbol": symbol,
        "interval": "1min",
        "extended_hours": "false",
        "outputsize": "full",
        "month": month,
    }
    params = urlencode(args)
    url = f'https://www.alphavantage.co/query?{params}'
    r = requests.get(url)
    if r.status_code in range(200, 299):
        return r.json()
    return None

In [10]:
# data = get_alpha_vantage_company_stock_quotes(symbol="GOOG")

In [11]:
# data 

In [12]:
stock_market_data_mapping = {
    '1. open': 'open_price',
    '2. high': 'high',
    '3. low': 'low',
    '4. close': 'close_price', 
    '5. volume': 'volume'
}

def rename_sm_keys(og_dict):
    return {stock_market_data_mapping[k]: v for k,v in og_dict.items()}

In [13]:
tz = timezone("US/Eastern")

def timezone_as_datetime_obj(timezone_str, use_tz=True):
    as_datetime = datetime.strptime(timezone_str, "%Y-%m-%d %H:%M:%S")
    if not use_tz:
        return as_datetime
    return tz.localize(as_datetime)

In [18]:
def clean_stock_quote_data(raw_dataset, company_obj=None, time_series_key="Time Series (1min)"):
    if company_obj is None:
        return []
    dataset = raw_dataset[time_series_key]
    timestamps = dataset.keys()
    cleaned_dataset = []
    for timestamp in timestamps:
        sm_data_raw = dataset[timestamp]
        cleaned_data = rename_sm_keys(sm_data_raw)
        cleaned_data['time'] = timezone_as_datetime_obj(timestamp, use_tz=True)
        cleaned_data['company'] = company_obj
        cleaned_dataset.append(cleaned_data)
    return cleaned_dataset

In [19]:
def batch_insert_stock_quotes(cleaned_dataset, batch_size = 500):
    for i in range(0, len(cleaned_dataset), batch_size):
        batch_chunk = cleaned_dataset[i:i+batch_size]
        new_data = [StockQuote(**x) for x in batch_chunk]
        StockQuote.objects.bulk_create(new_data, ignore_conflicts=True)
    return True

In [20]:
def get_monthly_company_stock_quotes(symbol="GOOG", month=None):
    if month is None:
        month = datetime.now().strftime("%Y-%m")
    company_obj = get_company_obj(symbol=symbol)
    raw_dataset = get_alpha_vantage_company_stock_quotes(symbol=symbol, month=month)
    cleaned_dataset = clean_stock_quote_data(raw_dataset, company_obj=company_obj)
    batch_insert_stock_quotes(cleaned_dataset)
    return cleaned_dataset

In [23]:
cleaned_dataset = get_monthly_company_stock_quotes(symbol="GOOG", month="2024-01")
# cleaned_dataset

OperationalError: connection failed: connection to server at "127.0.0.1", port 5431 failed: FATAL:  the database system is in recovery mode

In [ ]:
len(cleaned_dataset)

In [ ]:
# StockQuote.objects.all().delete()

In [ ]:
qs = StockQuote.objects.all()
# for obj in qs:
#     print(obj.time, obj.open_price)

In [ ]:
qs.count()